# Sistemas inteligentes. Aplicaciones

## Práctica 4. Autocorrector

En esta práctica vas a implementar un autocorrector. Este sistema es capaz de detectar aquellas palabras que no están escritas correctamente, y reemplaza cada una de ellas por la palabra más probable que esté a una menor distancia de edición (siempre que esta distancia sea menor o igual a 2).

Posteriormente, deberás crear otra función que, dada una frase y su versión corregida, sea capaz de identificar todas las operaciones necesarias de edición para corregirla.

### Creación del diccionario de probabilidades de las palabras del corpus

Vas a comenzar creando el diccionario que necesitas a partir del corpus de entrada. En este caso, el corpus está formado por 21 libros escritos por Pío Baroja (están almacenados en la carpeta libros). Para leer todos ellos, recuerda que a través de la librería ```os``` puedes obtener una lista con todos sus nombres. Los libros están codificados mediante UTF-8, por lo que puedes leerlos con la siguiente instrucción: ```open('nombreArchivo','r',encoding='UTF-8')```.

A partir de ese corpus debes crear un diccionario con todas las palabras que existen en el vocabulario indicando, para cada una de ellas, pa probabilidad de aparición de esa palabra en el corpus. Como solo queremos trabajar con las palabras, debes eliminar del texto todos aquellos símbolos que no sean considerados palabras por una expresión regular (recuerda los elementos ```\w``` y ```\W``` de las expresiones regulares).

In [ ]:
import re
import os

corpus = {}
num_words = 0
for name_file in os.listdir("libros/"):
    book = ""
    with open("libros/" + name_file, "r", encoding="UTF-8") as file:
        book = file.read()
        book = book.lower()
        book = re.sub(r"[^\w\s]", "", book)
        words = book.split()
        num_words += len(words)
    
    for word in words:
        if word in corpus:
            corpus[word] += 1
        else:
            corpus[word] = 1
            
for word, counter in corpus.items():
    corpus[word] /= num_words

### Funciones para aplicar una operación de edición
Ahora debes crear 4 funciones, cada una relacionada con una posible operación de edición: insertar, borrar, reemplazar e intercambiar. Cada una de estas funciones debe recibir una palabra y devolver una lista con todos los strings que se puedan obtener al aplicar esa operación de edición sobre la palabra recibida.

In [2]:
def insertar(word: str) -> list:
    new_words = []
    letters = list("abcdefghijklmnñopqrstuvwxyz")
    for letter in letters:
        for pos in range(len(word)):
            new_word = word[:pos] + letter + word[pos:]
            new_words.append(new_word)
    return new_words

In [3]:
def borrar(word: str) -> list:
    new_words = []
    for pos in range(len(word)):
        new_word = word[:pos] + word[pos + 1:]
        new_words.append(new_word)
    return new_words

In [4]:
def reemplazar(word: str) -> list:
    new_words = []
    letters = list("abcdefghijklmnñopqrstuvwxyz")
    for letter in letters:
        for pos in range(len(word)):
            new_word = word[:pos] + letter + word[pos + 1:]
            new_words.append(new_word)
    return new_words

In [5]:
def intercambiar(word: str) -> list:
    new_words = {}
    for pos in range(len(word) - 1):
        new_word = list(word)
        aux = new_word[pos + 1]
        new_word[pos + 1] = new_word[pos]
        new_word[pos] = aux
        new_words.append("".join(new_word))
    return new_words

### Función una_edicion
Utilizando las 4 funciones anteriores, crea una nueva función que, recibiendo una palabra, devuelva todos lo strings que se encuentran a distancia 1 de edición de dicha palabra.

Además de la palabra, esta función debe recibir como argumento el parámetro *intercambiar*, que por defecto vale *False*. Si este parámetro es verdadero, utilizamos las 4 operaciones de edición (insertar, borrar, reemplazar e intercambiar). Si este parámetro es falso, entonces solo utilizamos como operaciones de edición insertar, borrar y reemplazar.

Ten en cuenta que al aplicar dos operaciones de edición por separado, podríamos obtener un mismo string. Debes asegurarte de que en los strings que devuelve tu función no hay ninguno repetido.

In [6]:
def una_edicion(word: str, intercambiar: bool=False) -> set:
    words_insert = insertar(word)
    words_remove = borrar(word)
    words_replace = reemplazar(word)
    words_swap = intercambiar(word) if intercambiar else []
    words = set(words_insert + words_remove + words_replace + words_swap)
    return words

### Función dos_ediciones

Crea una función que recibe una palabra y devuelve todos los strings que se encuentran a 2 unidades de distancia de edición de esa palabra. Al igual que en el caso anterior, utiliza un pa´rametro intercambiar para permitir usar o no este tipo de operación de edición.

Vuelve a tener en cuenta que no debes devolver strings repetidos, aunque hayan sido creados aplicando diferentes operaciones de edición.

In [ ]:
def dos_ediciones(word: str, intercambiar: bool=False) -> set:
    words_dos_ediciones = set()
    words_una_edicion = una_edicion(word, intercambiar)
    for word_una_ed in words_una_edicion:
        words_dos_ediciones.union(una_edicion(word_una_ed))
    return words_dos_ediciones

### Función corregir

Utilizando las funciones anteriores, crea una nueva función que recibe una palabra y devuelve la o las correcciones sugeridas. Esta función recibe como argumentos la palabra a corregir, el diccionario de probabilidades de cada palabra del corpus, un valor entero (n) que indica el número máximo de correcciones sugeridas a devolver y un valor booleano que indica si queremos utilizar la operación de edición intercambiar o no.

Según la palabra recibida, esta función debe devolver:
* si la palabra existe en el diccionario, la devuelve tal cual
* si no
    * si existen palabras reales a distancia una, devuelve las n más probables, en orden de mayor a menor
    * si no
           * si existen palabras reales a distancia dos, devuelve las n más probables, en orden de mayor a menor
           * si no, devuelve la palabra original

El parámetro del número máximo de palabras a devolver se utiliza solo entre palabras que estén a la misma distancia de edición. Por ejemplo, si n=10 y existen 2 palabras a distancia de edición 1 y 20 palabras a distancia de edición 2, la función solo devolverá las dos palabras a distancia 1. Si, en otro caso, n=10 y existen 15 palabras a distancia de edición 1 y 30 palabras a distancia de edición 2, la función devolverá las 10 palabras más probables que están a distancia de edición 1.

In [35]:
def corregir(word: str, corpus: dict, max_correciones: int, intercambiar: bool=False) -> list:
    if word in corpus:
        return [word]
    
    for words_posibles in [una_edicion(word, intercambiar), dos_ediciones(word, intercambiar)]:
        validas = {}
        for word in words_posibles:
            if word in corpus:
                validas[word] = corpus[word]
        validas_sort = sorted(validas.items(), key=lambda x: x[1], reverse=True)
    
        if len(validas_sort) > 0:
            words = []
            minimo = min(len(validas_sort), max_correciones)
            for tupla in validas_sort[:minimo]:
                words.append(tupla[0])
            return words
    return [word]

### Ejecución

Ejecuta las funciones anteriores para, leída una palabra del usuario, devuelva las (máximo 5) palabras reales más probables a menor distancia de edición.

In [36]:
corregir("hechar", corpus, max_correciones=5)

['echar', 'hecha', 'hechas']

### Corrección de una frase

En este caso vamos a trabajar con una frase completa. Para cada una de las palabras que no sean correctas, debes cambiarlas por la palabra que esté a menor distancia de edición que sea más probable. Una vez que tienes la frase corregida, llama a otra función que calcule el número mínimo de operaciones de edición necesarias para pasar de la frase original a la corregida.

En este caso, fijamos que no se puede utilizar la operación de intercambiar y los costes de las otras tres operaciones los pasamos como parámetro a la función que calcula la distancia de edición.

Para calcular la distancia de edición mínima, utiliza el algoritmo de programación dinámica, y devuelve tanto la distancia mínima como la matriz calculada.

In [ ]:
import numpy as np

def calcular_coste_edicion_palabra(
        palabra_original: str,
        palabra_corregida: str,
        costes_ediciones:
        dict[str, int]
    ) -> tuple[int, np.ndarray]:
    
    coste_insertar = costes_ediciones["insertar"]
    coste_borrar = costes_ediciones["borrar"]
    coste_reemplazar = costes_ediciones["reemplazar"]

    matriz = np.zeros((
        len(palabra_original) + 1,
        len(palabra_corregida) + 1
    ))
    idx_filas = np.arange(0, matriz.shape[0])
    idx_cols = np.arange(0, matriz.shape[1])
    
    matriz[:, 0] = idx_filas
    matriz[0, :] = idx_cols

    for i in range(1, matriz.shape[0]):
        for j in range(1, matriz.shape[1]):
            if palabra_original[i - 1] == palabra_corregida[j - 1]:
                matriz[i, j] = matriz[i - 1, j - 1]
            else:
                costes_manipulaciones = [coste_borrar, coste_insertar, coste_reemplazar]
                costes_anteriores = [matriz[i - 1, j], matriz[i, j - 1], matriz[i - 1, j - 1]]
                pos = np.argmin(costes_anteriores)
                matriz[i, j] = costes_anteriores[pos] + costes_manipulaciones[pos]
    return matriz[-1, -1], matriz

def calcular_coste_edicion(frase_original: str, frase_corregida: str, costes_ediciones: dict[str, int]) -> int:
    matrices = []
    coste_total = 0
    for original, corregida in zip(frase_original.split(), frase_corregida.split()):
        coste, matriz = calcular_coste_edicion_palabra(original, corregida, costes_ediciones)
        coste_total += coste
        matrices.append(matriz)
    return coste_total

def corregir_frase(frase: str):
    frase_corregida = []
    for palabra in frase.split():
        palabras_posibles = corregir(palabra, corpus, max_correciones=5)
        palabra_corregida = palabras_posibles[0]
        frase_corregida.append(palabra_corregida)
    frase_corregida = ' '.join(frase_corregida)

    costes_ediciones = {
        "insertar": 1,
        "borrar": 1,
        "reemplazar": 1
    }
    return frase_corregida, calcular_coste_edicion(frase, frase_corregida, costes_ediciones)

### Todas las posibles ediciones
Para finalizar, escribe una función que recibe dos palabras (la original y la corregida) y la matriz de distancias mínimas calculadas mediante el algoritmo de programación dinámica. Esta función debe devolver la lista de ediciones que se debe hacer sobre la palabra original para conseguir la palabra corregida.

Como puede haber varios conjuntos de ediciones que obtengan la palabra corregida con el mísmo número de ediciones, debes devolver todos ellos.

In [61]:
def combinar_soluciones(caminos_hijos: list[list[str]], operacion_actual: str) -> list[list[str]]:
    caminos_actuales = []
    if len(caminos_hijos) == 0:
        caminos_actuales.append([operacion_actual])
    else:
        for j in range(len(caminos_hijos)):
            camino = caminos_hijos[j]
            camino.append(operacion_actual)
        caminos_actuales.extend(caminos_hijos)
    return caminos_actuales

def calcular_posibles_ediciones(
        palabra_original: str,
        palabra_corregida: str,
        matriz: np.ndarray,
        fila: int,
        col: int
    ) -> list[list[str]]:

    if fila == 0 and col == 0:
        return []
    else:
        caminos_actuales = []
        if palabra_original[fila - 1] == palabra_corregida[col - 1]:
            caminos_hijos = calcular_posibles_ediciones(palabra_original, palabra_corregida, matriz, fila - 1, col - 1)
            caminos_actuales.extend(caminos_hijos)
        else:
            minimo = np.min([matriz[fila - 1, col], matriz[fila, col - 1], matriz[fila - 1][col - 1]])
            if minimo == matriz[fila - 1, col]:
                operacion_actual = "Borrar " + palabra_original[fila - 1]
                caminos_hijos = calcular_posibles_ediciones(palabra_original, palabra_corregida, matriz, fila - 1, col)
                caminos = combinar_soluciones(caminos_hijos, operacion_actual)
                caminos_actuales.extend(caminos)
            if minimo == matriz[fila, col - 1]:
                operacion_actual = "Insertar " + palabra_corregida[col - 1]
                caminos_hijos = calcular_posibles_ediciones(palabra_original, palabra_corregida, matriz, fila, col - 1)
                caminos = combinar_soluciones(caminos_hijos, operacion_actual)
                caminos_actuales.extend(caminos)
            if minimo == matriz[fila - 1, col - 1]:
                operacion_actual = "Reemplazar " + palabra_original[fila - 1] + "->" + palabra_corregida[col - 1]
                caminos_hijos = calcular_posibles_ediciones(palabra_original, palabra_corregida, matriz, fila - 1, col - 1)
                caminos = combinar_soluciones(caminos_hijos, operacion_actual)
                caminos_actuales.extend(caminos)
        return caminos_actuales
    
palabra_original = "partir"
palabra_corregida = "tripa"
costes_ediciones = {
    "insertar": 1,
    "borrar": 1,
    "reemplazar": 1
}
coste, matriz = calcular_coste_edicion_palabra(palabra_original, palabra_corregida, costes_ediciones)
caminos_posibles = calcular_posibles_ediciones(palabra_original, palabra_corregida, matriz, matriz.shape[0] - 1, matriz.shape[1] - 1)
print(len(caminos_posibles))
for camino in caminos_posibles:
    print(camino)

6
['Reemplazar p->t', 'Borrar a', 'Borrar t', 'Reemplazar r->p', 'Insertar a']
['Borrar p', 'Reemplazar a->t', 'Borrar t', 'Reemplazar r->p', 'Insertar a']
['Reemplazar p->t', 'Borrar a', 'Borrar t', 'Insertar p', 'Reemplazar r->a']
['Borrar p', 'Reemplazar a->t', 'Borrar t', 'Insertar p', 'Reemplazar r->a']
['Reemplazar p->t', 'Borrar a', 'Reemplazar t->i', 'Reemplazar i->p', 'Reemplazar r->a']
['Borrar p', 'Reemplazar a->t', 'Reemplazar t->i', 'Reemplazar i->p', 'Reemplazar r->a']
